# Problem: Personalized Coupon Recommendation Using User Similarity

ShopSmart wants to improve customer engagement by recommending personalized coupons based on the coupon usage patterns of similar customers.

## Coupon Types

The platform offers five coupon types:

1. **10% off Electronics**
2. **Free Shipping**
3. **Buy One Get One Free**
4. **Weekend Special**
5. **$10 off Discount**

Each customer is represented by a **5-element binary vector**:

* `1` → the customer has used the coupon
* `0` → the customer has not used the coupon

For example:

```python
[0, 0, 1, 0, 0]
```

means the customer has used only the third coupon, **Buy One Get One Free**.

---

## Similarity Function

Use Euclidean distance to calculate similarity between two customer vectors:

$$
\text{similarity}(u,v)
=
\frac{1}{1 + \text{EuclideanDistance}(u,v)}
$$

where:

$$
\text{EuclideanDistance}(u,v)
=
\sqrt{\sum_i (u_i-v_i)^2}
$$

A larger similarity score indicates that two customers have more similar coupon usage patterns.

---

## Recommendation Algorithm

Given a target customer:

1. Calculate the Euclidean similarity between the target customer and every existing customer.
2. Select the **top 5 most similar customers**.
3. For every coupon that the target customer has **not yet used**, calculate its recommendation score:

$$
\text{CouponScore}(j)
=
\sum_{i \in TopK}
\text{Similarity}(i,\text{target}) \times \text{Usage}_{i,j}
$$

where:

* `Similarity(i, target)` is the similarity between customer `i` and the target customer.
* `Usage(i, j)` is `1` if customer `i` has used coupon `j`, otherwise `0`.

4. Exclude coupons already used by the target customer.
5. Rank the remaining coupons by recommendation score in **descending order**.
6. If multiple coupons have the same score, any valid ordering among the tied coupons is acceptable.

---

## Example

Suppose the target customer is:

```python
target = [0, 0, 1, 0, 0]
```

and the three most similar customers are:

```text
[0, 0, 1, 1, 0]  -> similarity = 0.5000
[1, 0, 1, 0, 1]  -> similarity = 0.4142
[0, 0, 1, 1, 1]  -> similarity = 0.4142
```

The score for **Weekend Special** is:

$$
(1 \times 0.5000)
+
(0 \times 0.4142)
+
(1 \times 0.4142)
=
0.9142
$$

The third coupon is not considered because the target customer has already used it.

---

## Input

```python
existing_customers = [
    [1, 0, 1, 0, 1],  # User 1
    [0, 1, 0, 1, 0],  # User 2
    [1, 1, 1, 0, 0],  # User 3
    [0, 0, 1, 1, 1],  # User 4
    [1, 0, 0, 1, 0],  # User 5
    [0, 1, 0, 0, 1],  # User 6
    [1, 1, 0, 1, 1],  # User 7
    [0, 0, 1, 1, 0],  # User 8
    [1, 1, 1, 1, 0],  # User 9
    [0, 1, 1, 0, 1],  # User 10
]

target = [0, 0, 1, 0, 0]
```

---

## Expected Output

Using the **top 5 most similar customers**:

```text
Recommended Coupons with Scores:

- $10 off Discount: 1.24
- Weekend Special: 0.91
- 10% off Electronics: 0.83
- Free Shipping: 0.83
```

## Task

Implement a function that takes:

* the existing customer vectors,
* the target customer's vector,
* and optionally `top_k`,

and returns the unused coupons ranked by their recommendation scores in descending order.


In [11]:

#functional implementation*

existing_customers = [

  [1, 0, 1, 0, 1],  # User 1*

  [0, 1, 0, 1, 0],  # User 2*

  [1, 1, 1, 0, 0],  # User 3*

  [0, 0, 1, 1, 1],  # User 4*

  [1, 0, 0, 1, 0],  # User 5*

  [0, 1, 0, 0, 1],  # User 6*

  [1, 1, 0, 1, 1],  # User 7*

  [0, 0, 1, 1, 0],  # User 8*

  [1, 1, 1, 1, 0],  # User 9*

  [0, 1, 1, 0, 1]  # User 10*  

]

target = [0, 0, 1, 0, 0]






In [ ]:
import numpy as np


# Function to calculate similarity using Euclidean distance
def euclidean_similarity(vec1, vec2):
    """
    Similarity = 1 / (1 + Euclidean distance)

    Higher similarity means users are more similar.
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    distance = np.linalg.norm(vec1 - vec2)
    return 1 / (1 + distance)


coupon_names = [
    "10% off Electronics",
    "Free Shipping",
    "Buy One Get One Free",
    "Weekend Special",
    "$10 off Discount"
]


existing_customers = [
    [1, 0, 1, 0, 1],  # User 1
    [0, 1, 0, 1, 0],  # User 2
    [1, 1, 1, 0, 0],  # User 3
    [0, 0, 1, 1, 1],  # User 4
    [1, 0, 0, 1, 0],  # User 5
    [0, 1, 0, 0, 1],  # User 6
    [1, 1, 0, 1, 1],  # User 7
    [0, 0, 1, 1, 0],  # User 8
    [1, 1, 1, 1, 0],  # User 9
    [0, 1, 1, 0, 1],  # User 10
]

target = [0, 0, 1, 0, 0]


def recommend_coupons(existing_customers, target, top_k=5):
    customers = np.array(existing_customers)
    target = np.array(target)

    # 1. Calculate similarity between target and every customer
    similarities = np.array([
        euclidean_similarity(user, target)
        for user in customers
    ])

    # 2. Get top-K most similar customers
    top_indices = np.argsort(similarities)[::-1][:top_k]

    top_users = customers[top_indices]
    top_similarities = similarities[top_indices]

    # 3. Weight each user's coupon vector by that user's similarity
    #
    # top_users shape:        (top_k, num_coupons)
    # top_similarities shape: (top_k,)
    #
    # [:, None] changes it to (top_k, 1), so each row/user
    # gets multiplied by its corresponding similarity.
    weighted_users = top_users * top_similarities[:, None]

    # 4. Sum contributions for each coupon
    coupon_scores = weighted_users.sum(axis=0)

    # 5. Keep only coupons that target user has NOT used
    recommendations = []

    for i, used in enumerate(target):
        if used == 0:
            recommendations.append(
                (coupon_names[i], coupon_scores[i])
            )

    # 6. Rank coupons by score descending
    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations


recommendations = recommend_coupons(
    existing_customers,
    target,
    top_k=5
)


print("Recommended Coupons with Scores:\n")

for coupon, score in recommendations:
    print(f"- {coupon}: {score:.2f}")